### lấy metric

In [1]:
from confluent_kafka import Consumer, KafkaError
from time import monotonic_ns, sleep
from datetime import datetime
import json
import os

stats_file_path = f"E:/UNIVERSITY_SUBJECTS/projects/E_Commercial_Data_Pipeline/evidence/w03"
os.makedirs(stats_file_path, exist_ok=True)

def stats_cb(stats_json):
    stats = json.loads(stats_json)
    timestamp = stats.get("ts")
    rxmsgs = stats.get("rxmsgs")
    rxmsg_bytes = stats.get("rxmsg_bytes")
    # print(f"Stats: timestamp={timestamp}, time={time}, rxmsgs={rxmsgs}, rxmsg_bytes={rxmsg_bytes}")
    with open(f"{stats_file_path}/consumer_stats.csv", "a", encoding="utf-8", newline="") as f:
        f.write(f"{timestamp},{rxmsgs},{rxmsg_bytes}\n")
    f.close()
    print(stats)
    
consumer = Consumer({
    'bootstrap.servers': 'kafka-1:9092',
    'group.id': 'evaluate-consumer-group',
    'group.instance.id': 'evaluate-consumer-group-instance-1',
    'auto.offset.reset': 'earliest', 
    'enable.auto.commit': False,
    'enable.auto.offset.store': False,
    'statistics.interval.ms': 1,
    'debug': 'broker,protocol,fetch',
    'error_cb': lambda err: print(f"Error: {err}"),
    'stats_cb': stats_cb,
})

consumer.subscribe(['product_data_changes'])

idx = 0
to_stop = 0
metric_records = []

with open(f"{stats_file_path}/consumer_error.csv", 'a') as f:
    while(True):
        # time1 = monotonic_ns()        
        msg = consumer.poll(0.05)
        # time2 = monotonic_ns()   
        # latency = time2 - time1
        idx += 1
        
        if msg is None:
            print("No message received. Waiting...")
            to_stop += 1
            if to_stop >= 100:
                break
            continue
        elif msg.error():
            print(f"Error: {msg.error()}")
            metric_records.append(f"{msg.offset()},{msg.partition()},{msg.topic()},error")
        else:   
            metric_records.append(f"{msg.offset()},{msg.partition()},{msg.topic()},success")
            # consumer.store_offsets(msg)
            # consumer.commit(msg)

        if idx == 10:
            #thứ tự các cột: offset,partition,topic,latency,status
            for record in metric_records:
                f.write(record + "\n")
            metric_records = []
            idx = 0
    for record in metric_records:   
        f.write(record + "\n")

{'name': 'rdkafka#consumer-1', 'client_id': 'rdkafka', 'type': 'consumer', 'ts': 14781029494, 'time': 1792909338, 'age': 2444, 'replyq': 0, 'msg_cnt': 0, 'msg_size': 0, 'msg_max': 0, 'msg_size_max': 0, 'simple_cnt': 0, 'metadata_cache_cnt': 1, 'brokers': {'kafka-1:9092/bootstrap': {'name': 'kafka-1:9092/bootstrap', 'nodeid': -1, 'nodename': 'kafka-1:9092', 'source': 'configured', 'state': 'TRY_CONNECT', 'stateage': 174, 'outbuf_cnt': 0, 'outbuf_msg_cnt': 0, 'waitresp_cnt': 0, 'waitresp_msg_cnt': 0, 'tx': 0, 'txbytes': 0, 'txerrs': 0, 'txretries': 0, 'txidle': -1, 'req_timeouts': 0, 'rx': 0, 'rxbytes': 0, 'rxerrs': 0, 'rxcorriderrs': 0, 'rxpartial': 0, 'rxidle': -1, 'zbuf_grow': 0, 'buf_grow': 0, 'wakeups': 1, 'connects': 0, 'disconnects': 0, 'int_latency': {'min': 0, 'max': 0, 'avg': 0, 'sum': 0, 'stddev': 0, 'p50': 0, 'p75': 0, 'p90': 0, 'p95': 0, 'p99': 0, 'p99_99': 0, 'outofrange': 0, 'hdrsize': 0, 'cnt': 0}, 'outbuf_latency': {'min': 0, 'max': 0, 'avg': 0, 'sum': 0, 'stddev': 0, 'p

### Tính toán

In [2]:
import pandas as pd

stats_file_path = f"E:/UNIVERSITY_SUBJECTS/projects/E_Commercial_Data_Pipeline/evidence/w03"

df_1 = pd.read_csv(f"{stats_file_path}/consumer_stats.csv", names=["timestamp", "rxmsgs", "rxmsg_bytes"])
# df_1 = df_1.apply(pd.to_numeric, args=('coerce',))

In [3]:
df_1_2 = df_1.copy()

In [4]:
df_1_2.drop_duplicates(subset=["rxmsgs"], keep="last", inplace=True)

In [5]:
df_1.drop_duplicates(subset=["rxmsgs"], keep="first", inplace=True)

In [6]:
df = pd.concat([df_1, df_1_2], ignore_index=True)
df.sort_values(by="timestamp", ignore_index=True, inplace=True)
df.drop(0, inplace=True)
df = df.iloc[:-1]
df.reset_index(drop=True, inplace=True)

In [7]:
even_row_df = df.iloc[::2]
odd_row_df = df.iloc[1::2]
even_row_df.reset_index(drop=True, inplace=True)
odd_row_df.reset_index(drop=True, inplace=True)

In [8]:
substracted_df = odd_row_df - even_row_df

In [9]:
received_msgs = substracted_df['rxmsgs'].sum()

In [10]:
latency = substracted_df['timestamp'].mean()
latency = latency / 100000000

In [11]:
msg_throughput = substracted_df['rxmsgs'].sum() / latency
bytes_throughput = substracted_df['rxmsg_bytes'].sum() / latency

In [12]:
print(
    f"Received messages: {received_msgs}",
    f"Message throughput: {round(msg_throughput)} msg/s",
    f"Bytes throughput: {round(bytes_throughput, 2)} bytes/s",
    sep="\n"
)

Received messages: 784
Message throughput: 4830263 msg/s
Bytes throughput: 93187431458.32 bytes/s
